In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from torch import device, cuda, tensor, float32
from torch.utils.data import Dataset,DataLoader 
from torch.nn import BCEWithLogitsLoss
import torchvision.transforms as transform
from PIL import Image
from warnings import filterwarnings
filterwarnings("ignore")

%matplotlib inline

dev = device("cuda" if cuda.is_available() else "cpu")
dev

device(type='cuda')

In [3]:
df = pd.read_csv(r"D:\Data\Data_Entry_2017.csv")

In [ ]:
PATH = r"D:\Data"
IMG_FOLDERS = [os.path.join(PATH,x+"\\images") for x in os.listdir(PATH) if x.startswith("images_")] 

NUM_OF_CLASSES = 14

ALL_DISEASES = sorted(df[df["Finding Labels"] != "No Finding"]["Finding Labels"].str.split("|").explode().unique())
ALL_DISEASES

['Atelectasis',
 'Cardiomegaly',
 'Consolidation',
 'Edema',
 'Effusion',
 'Emphysema',
 'Fibrosis',
 'Hernia',
 'Infiltration',
 'Mass',
 'Nodule',
 'Pleural_Thickening',
 'Pneumonia',
 'Pneumothorax']

In [8]:
df_imgs_labels = df[["Image Index","Finding Labels"]]
df_imgs_labels

,Image Index,Finding Labels
0,00000001_000.png,Cardiomegaly
1,00000001_001.png,Cardiomegaly|Emphysema
2,00000001_002.png,Cardiomegaly|Effusion
3,00000002_000.png,No Finding
4,00000003_000.png,Hernia
...,...,...
112115,00030801_001.png,Mass|Pneumonia
112116,00030802_000.png,No Finding
112117,00030803_000.png,No Finding
112118,00030804_000.png,No Finding


In [ ]:
class ChestXRayDataset(Dataset):
    def __init__(self,df,img_folders,transforms=None):
        self.df = df.reset_index(drop=True)
        self.img_folders = img_folders
        self.transforms = transforms
        self.labels = self._encode_labels()

    def _encode_labels(self):
        encode = []

        for label in self.df["Finding Labels"]:
            diseases_in_img = label.str.split("|")

            vector = [0.0] * NUM_OF_CLASSES
            for disease in diseases_in_img:
                if disease in ALL_DISEASES:
                    idx = ALL_DISEASES.index(disease)
                    vector[idx] = 1.0
            encode.append(vector)
        return encode
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        img_name = self.df.loc[index,"Image Index"]

        img = None
        for folder in IMG_FOLDERS:
            img_path = os.path.join(folder,img_name)
            if os.path.exists(img_path):
                img = Image.open(img_path).convert("RGB")
                break
        
        if self.transforms:
            img = self.transforms(img)

        label = tensor(self.labels[index],dtype=float32)
        return img,label
    
dataset = ChestXRayDataset(df_imgs_labels,IMG_FOLDERS,)